In [0]:
import sys
import os

# Add the project directory to Python path
sys.path.insert(0, "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks")

# Import the ADLSManager class from ADLS_Databricks_Connection module
from ADLS_Databricks_Connection import ADLSManager

# Initialize ADLSManager with the trains ETL config
config_path = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config_trains.json"
manager = ADLSManager(config_path)

# Print configuration summary
manager.print_summary()

In [0]:
# Inject Databricks globals into the imported module
import ADLS_Databricks_Connection as adls_module
adls_module.dbutils = dbutils
adls_module.spark = spark

# Step 1: Configure OAuth for ADLS Gen2 access
manager.configure_oauth()

# Step 2: Read the JSON source file (EXP-TRAINS.json)
df_trains = manager.read_file
(
    file_path=manager.file_cfg["json"],
    file_format="json",
    multiLine=True
)

# Store in manager's dataframes dict for downstream use
manager.dataframes["json"] = df_trains

# Display the data
print(f"Row count: {df_trains.count()}")
print(f"Columns: {df_trains.columns}\n")
display(df_trains)

In [0]:
from pyspark.sql.functions import explode, col

# Explode the trainRoute array so each stop becomes its own row
df_stops = (
    df_trains
    .select
    (
        col("trainNumber"),
        col("trainName"),
        col("route"),
        col("runningDays.MON").alias("MON"),
        col("runningDays.TUE").alias("TUE"),
        col("runningDays.WED").alias("WED"),
        col("runningDays.THU").alias("THU"),
        col("runningDays.FRI").alias("FRI"),
        col("runningDays.SAT").alias("SAT"),
        col("runningDays.SUN").alias("SUN"),
        explode(col("trainRoute")).alias("stop")
    )
    .select
    (
        "trainNumber",
        "trainName",
        "route",
        "MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN",
        col("stop.sno").alias("stop_number"),
        col("stop.stationName").alias("station_name"),
        col("stop.arrives").alias("arrives"),
        col("stop.departs").alias("departs"),
        col("stop.day").alias("day"),
        col("stop.distance").alias("distance")
    )
)

print(f"Total stops: {df_stops.count()}")
print(f"Columns: {df_stops.columns}\n")
display(df_stops)

In [0]:
# Build the bronze ADLS path from config
bronze_path = manager.get_path(manager.config["medallion_paths"]["bronze"])
print(f"Bronze target: {bronze_path}")

# Write flattened stops DataFrame to bronze as Delta
df_stops.write.format("delta").mode("overwrite").save(bronze_path)
10:44 AM (3s)
4:
Write Flattened Data to Bronze Delta Layer
Write Flattened Data to Bronze Delta Layer
$0
print(f"\u2714 Flattened data written to bronze Delta layer")
print(f"  Rows: {df_stops.count()} | Columns: {len(df_stops.columns)}")

# Verify by reading back
df_bronze = spark.read.format("delta").load(bronze_path)
print(f"\u2714 Verification read-back: {df_bronze.count()} rows")
display(df_bronze)

In [0]:
from pyspark.sql.functions import 
(
    col, trim, upper, regexp_extract, split as spark_split,
    when, lit, current_timestamp, concat_ws
)
from pyspark.sql.types import IntegerType, DoubleType

# ------------------------------------------------------------------ #
#  Read bronze layer                                                   #
# ------------------------------------------------------------------ #
bronze_path = manager.get_path(manager.config["medallion_paths"]["bronze"])
df_bronze = spark.read.format("delta").load(bronze_path)
print(f"Bronze rows: {df_bronze.count()}")

# ------------------------------------------------------------------ #
#  Cleansing & Transformation                                          #
# ------------------------------------------------------------------ #
df_silver = 
(
    df_bronze
    # 1. Trim whitespace on string columns
    .withColumn("trainNumber", trim(col("trainNumber")))
    .withColumn("trainName",   trim(col("trainName")))
    .withColumn("route",       trim(col("route")))
    .withColumn("station_name", trim(col("station_name")))

    # 2. Split station_name "MARWAR JN - MJ" → name + code
    .withColumn("station_code",
        trim(regexp_extract(col("station_name"), r"- (.+)$", 1)))
    .withColumn("station_name_clean",
        trim(regexp_extract(col("station_name"), r"^(.+?) -", 1)))

    # 3. Split route into origin and destination
    .withColumn("origin",
        trim(regexp_extract(col("route"), r"^(.+?) to ", 1)))
    .withColumn("destination",
        trim(regexp_extract(col("route"), r" to (.+)$", 1)))

    # 4. Cast stop_number and day to integer
    .withColumn("stop_number", col("stop_number").cast(IntegerType()))
    .withColumn("day",         col("day").cast(IntegerType()))

    # 5. Parse distance: "25 kms" → 25.0
    .withColumn("distance_km",
        regexp_extract(col("distance"), r"(\d+)", 1).cast(DoubleType()))

    # 6. Flag source/destination stops and normalize arrival/departure
    .withColumn("is_source",      col("arrives") == "Source")
    .withColumn("is_destination", col("departs") == "Destination")
    .withColumn("arrival_time",
        when(col("arrives") == "Source", lit(None)).otherwise(col("arrives")))
    .withColumn("departure_time",
        when(col("departs") == "Destination", lit(None)).otherwise(col("departs")))

    # 7. Add audit column
    .withColumn("ingested_at", current_timestamp())

    # 8. Select final silver schema
    .select(
        "trainNumber", "trainName", "route", "origin", "destination",
        "MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN",
        "stop_number", "station_name_clean", "station_code",
        "arrival_time", "departure_time", "day", "distance_km",
        "is_source", "is_destination", "ingested_at"
    )
    .withColumnRenamed("station_name_clean", "station_name")
)

# ------------------------------------------------------------------ #
#  Quarantine: rows with critical nulls                                #
# ------------------------------------------------------------------ #
quarantine_condition = (
    col("trainNumber").isNull() |
    col("station_name").isNull() | (col("station_name") == "") |
    col("stop_number").isNull() |
    col("distance_km").isNull()
)

df_quarantine = df_silver.filter(quarantine_condition)
df_silver_clean = df_silver.filter(~quarantine_condition)

print(f"Silver clean rows : {df_silver_clean.count()}")
print(f"Quarantine rows   : {df_quarantine.count()}")

display(df_silver_clean)

In [0]:
# ------------------------------------------------------------------ #
#  Write silver clean data                                             #
# ------------------------------------------------------------------ #
silver_path = manager.get_path(manager.config["medallion_paths"]["silver"])
df_silver_clean.write.format("delta").mode("overwrite").save(silver_path)
print(f"\u2714 Silver written to: {silver_path}")

# ------------------------------------------------------------------ #
#  Write quarantine data                                               #
# ------------------------------------------------------------------ #
quarantine_path = manager.get_path(manager.config["medallion_paths"]["quarantine"])
df_quarantine.write.format("delta").mode("overwrite").save(quarantine_path)
print(f"\u2714 Quarantine written to: {quarantine_path}")

# ------------------------------------------------------------------ #
#  Verify read-back                                                    #
# ------------------------------------------------------------------ #
silver_count = spark.read.format("delta").load(silver_path).count()
quarantine_count = spark.read.format("delta").load(quarantine_path).count()

print(f"\n{'='*50}")
print(f"  Silver Layer Summary")
print(f"{'='*50}")
print(f"  Bronze input      : {df_bronze.count():,} rows")
print(f"  Silver clean      : {silver_count:,} rows")
print(f"  Quarantine        : {quarantine_count:,} rows")
print(f"  Total accounted   : {silver_count + quarantine_count:,} rows")
print(f"{'='*50}")

In [0]:
from pyspark.sql.functions import 
(
    col, count, countDistinct, max as spark_max, min as spark_min,
    sum as spark_sum, round as spark_round, when, lit
)

# ------------------------------------------------------------------ #
#  Read silver layer                                                   #
# ------------------------------------------------------------------ #
silver_path = manager.get_path(manager.config["medallion_paths"]["silver"])
df_silver = spark.read.format("delta").load(silver_path)

# ------------------------------------------------------------------ #
#  Gold: Route Summary Aggregation                                     #
# ------------------------------------------------------------------ #
df_gold_route = (
    df_silver
    .groupBy("trainNumber", "trainName", "route", "origin", "destination")
    .agg(
        count("stop_number").alias("total_stops"),
        spark_max("distance_km").alias("total_distance_km"),
        spark_max("stop_number").alias("max_stop_number"),
        countDistinct("station_name").alias("unique_stations"),
        spark_max("day").alias("journey_days"),

        # Running days: take first (all rows per train are the same)
        spark_max(col("MON").cast("int")).alias("MON"),
        spark_max(col("TUE").cast("int")).alias("TUE"),
        spark_max(col("WED").cast("int")).alias("WED"),
        spark_max(col("THU").cast("int")).alias("THU"),
        spark_max(col("FRI").cast("int")).alias("FRI"),
        spark_max(col("SAT").cast("int")).alias("SAT"),
        spark_max(col("SUN").cast("int")).alias("SUN"),
    )
    # Compute days per week the train runs
    .withColumn("days_per_week",
        col("MON") + col("TUE") + col("WED") +
        col("THU") + col("FRI") + col("SAT") + col("SUN")
    )
    # Average distance between stops
    .withColumn("avg_km_per_stop",
        spark_round(col("total_distance_km") / col("total_stops"), 2)
    )
    .orderBy(col("total_distance_km").desc())
)

print(f"Gold route summary rows: {df_gold_route.count()}")
display(df_gold_route)

In [0]:
# ------------------------------------------------------------------ #
#  Write gold route summary                                            #
# ------------------------------------------------------------------ #
gold_route_path = manager.get_path(manager.config["medallion_paths"]["gold_route_summary"])
df_gold_route.write.format("delta").mode("overwrite").save(gold_route_path)

# Verify
gold_count = spark.read.format("delta").load(gold_route_path).count()

print(f"\u2714 Gold route summary written to: {gold_route_path}")
print(f"  Rows: {gold_count:,}")
print(f"  Columns: {len(df_gold_route.columns)}")